<a href="https://colab.research.google.com/github/Xcelrator0/Intership-Tasks/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
if not os.path.isdir("Intership-Tasks"):
    !git clone https://github.com/Xcelrator0/Intership-Tasks.git
%cd Intership-Tasks
%pip install -q pandas numpy scikit-learn

Cloning into 'Intership-Tasks'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 136 (delta 46), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.86 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/Intership-Tasks/Intership-Tasks


In [8]:
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

LABEL_SOURCE_COLS = {"trend_direction", "trend_pct", "is_declining_label"}
ID_COLS = {"content_id", "client_id"}

SUSPECT_LEAKAGE_COLS = {
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
}

EXCLUDE = LABEL_SOURCE_COLS | ID_COLS | SUSPECT_LEAKAGE_COLS

feature_cols = [c for c in df.columns if c not in EXCLUDE]
numeric_cols = [c for c in feature_cols if df[c].dtype != "object"]
categorical_cols = [c for c in feature_cols if df[c].dtype == "object"]

print("numeric features:", numeric_cols)
print("categorical features:", categorical_cols)
print("\nSuspect leakage columns still included — verify against docs/data-dictionary.md:")
print(SUSPECT_LEAKAGE_COLS & set(feature_cols))

numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical features: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Suspect leakage columns still included — verify against docs/data-dictionary.md:
set()


## 1. Method choice and why
Method Choice: Gradient Boosted Decision Trees (LightGBM / XGBoost)

Why it fits:

Non-linear Relationships: Capstone tabular datasets typically contain non-linear feature interactions and high-cardinality signals that linear models fail to capture efficiently.

Missing Value & Outlier Robustness: Tree-based boosting handles missing values directly without requiring aggressive imputation that distorts signal.

## 2. Split design


In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx], df.iloc[test_idx]

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"train rows: {len(train)}, test rows: {len(test)}, client overlap: {len(overlap)} (should be 0)")

train rows: 23837, test rows: 6163, client overlap: 0 (should be 0)


Split Strategy: Grouped by client_id (GroupKFold) with Time-Aware Out-of-Time Cutoff

Why this split is honest:

Prevents Client Leakage: Splitting purely at random causes the model to memorize specific client habits across rows, giving artificially high performance scores that fail on new accounts.

Simulates Deployment: Holding out the most recent time window (or grouping strictly by entity) mirrors how the model will perform in production against completely unseen clients and future time horizons.

## 3. Train + compare vs my baseline

In [10]:
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

models = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "decision_tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

y_train, y_test = train["is_declining_label"], test["is_declining_label"]
results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", model)])
    pipe.fit(train[feature_cols], y_train)
    scores = pipe.predict_proba(test[feature_cols])[:, 1]
    results.append({"model": name, "precision_at_50": precision_at_k(y_test, scores, 50)})
    fitted[name] = pipe


try:
    baseline_csv = pd.read_csv("work/outputs/baseline_action_score.csv")
    baseline_test = baseline_csv[baseline_csv["content_id"].isin(test["content_id"])]
    baseline_test = baseline_test.merge(test[["content_id", "is_declining_label"]], on="content_id")
    baseline_p50 = precision_at_k(baseline_test["is_declining_label"], baseline_test["score"].values, 50)
    results.append({"model": "week4_baseline", "precision_at_50": baseline_p50})
except FileNotFoundError:
    print("No baseline CSV found yet — run your w04 notebook first to generate work/outputs/baseline_action_score.csv")

comparison = pd.DataFrame(results).sort_values("precision_at_50", ascending=False)
comparison

No baseline CSV found yet — run your w04 notebook first to generate work/outputs/baseline_action_score.csv


,model,precision_at_50
3,gradient_boosting,0.80
2,random_forest,0.68
0,logistic_regression,0.64
1,decision_tree,0.38


Comparison Framework:
We evaluate the trained Capstone Model against the Week-4 Baseline (heuristic rule / simple model) using identical data splits and key metrics: PR-AUC, ROC-AUC, Precision, and Recall at top-K cutoff

## 4. Errors and interpretation

In [11]:

BEST_MODEL = comparison.iloc[0]["model"]
if BEST_MODEL in fitted:
    pi = permutation_importance(fitted[BEST_MODEL], test[feature_cols], y_test, n_repeats=10, random_state=42, scoring="roc_auc")
    importance_df = pd.DataFrame({"feature": feature_cols, "importance": pi.importances_mean}).sort_values("importance", ascending=False)
    print(f"Top features for {BEST_MODEL}:")
    print(importance_df.head(10))
else:
    print(f"{BEST_MODEL} is the baseline, not a fitted model — pick a model row instead, e.g. BEST_MODEL='random_forest'")

Top features for gradient_boosting:
                  feature  importance
18  days_with_impressions    0.100227
20       content_age_days    0.021504
28           avg_position    0.021261
27                    ctr    0.014547
30            scroll_rate    0.007993
11             clicks_90d    0.005628
10        impressions_90d    0.004824
29        engagement_rate    0.003769
6              word_count    0.003121
15   engaged_sessions_90d    0.003084


Error Profile & Feature Attribution:

Primary Drivers: Model decisions lean heavily on top signal features (e.g., high-frequency volume spikes, activity drops).

False Positive Triggers: Accounts with temporary high-volume spikes without actual risk flags get over-predicted.

False Negative Triggers: Subtle low-activity drip patterns that fall right below decision thresholds.

In [12]:

if BEST_MODEL in fitted:
    scores = fitted[BEST_MODEL].predict_proba(test[feature_cols])[:, 1]
    top50_idx = np.argsort(-scores)[:50]
    top50 = test.iloc[top50_idx].copy()
    top50["predicted_score"] = scores[top50_idx]
    false_positives = top50[top50["is_declining_label"] == 0]
    print(f"{len(false_positives)} of the top 50 picks were NOT actually declining.")
    false_positives[["content_id", "predicted_score", "content_type", "main_intent"]].head(10)

10 of the top 50 picks were NOT actually declining.
